# T-KGAT Stage 1 — thời gian trên cạnh user–item

Dùng nguyên Dataset code/data và ZIP Recovery **KGAT đã hoàn thành 20/20 lượt,
có kết quả test đủ 5 seed**. Không cần chờ KGAT không KG.
Notebook chỉ chạy `tkgat.yaml`, tự đọc cấu hình KGAT đã chọn trên validation.
Quét đủ **γ = {1, 3, 10} × 5 seed 2019–2023 = 15 lượt**.
Chọn γ theo Recall@20 trung bình trên validation; chỉ cấu hình được chọn mới chấm test.

Mặc định chạy tối đa 2 lượt mới mỗi phiên, lưu ZIP sau mỗi lượt hoàn tất.
Chọn **Save Version → Save & Run All** với GPU T4. Chờ log
`Running tkgat Stage 1 only` rồi có thể đóng tab/tắt máy.

In [ ]:
from pathlib import Path

PAYLOAD_DIR = None
# Select a ZIP or results directory only if automatic discovery finds multiple recoveries.
RESTORE_SOURCE = None
PROJECT_ROOT = Path('/kaggle/working/t-kgat-stage1')
RUN_TRAINING = True
NEW_TKGAT_FITS_THIS_SESSION = 2
CHECKPOINT_ZIP = Path('/kaggle/working/tkgat-lastfm-stage1-results.zip')

## 1. Kiểm tra source/data và khôi phục KGAT

Phiên đầu dùng `tkgat-lastfm-results.zip` hoàn chỉnh của KGAT có KG.
Phiên sau dùng ZIP `tkgat-lastfm-stage1-results.zip` mới nhất của chính nhánh Stage 1.
Giữ một bản Recovery trong Input; nếu có nhiều bản, chỉ định `RESTORE_SOURCE`.
ZIP hoặc thư mục `results/` đã giải nén đều được hỗ trợ.

In [ ]:
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile, is_zipfile


def recovery_archive(input_root, working, expected_identity, explicit=None):
    """Resolve one matching ZIP or extracted results directory, never guess between copies."""
    if explicit is not None:
        location = Path(explicit)
        if not location.exists():
            raise RuntimeError(f'Recovery path does not exist: {location}')
        locations = [location]
    else:
        locations = sorted(input_root.rglob('*.zip'))
        locations += sorted(p.parent for p in input_root.rglob('kaggle-run.json')
                            if p.parent.name == 'results')
    matches = []
    inspected = []
    for location in locations:
        if location.is_file() and is_zipfile(location):
            with ZipFile(location) as archive:
                marker = 'results/kaggle-run.json'
                if marker not in archive.namelist():
                    continue
                saved_identity = json.loads(archive.read(marker))
            candidate = ('zip', location)
        elif location.is_dir():
            folder = location if location.name == 'results' else location / 'results'
            marker = folder / 'kaggle-run.json'
            if not marker.is_file():
                continue
            saved_identity = json.loads(marker.read_text(encoding='utf-8'))
            candidate = ('directory', folder)
        else:
            continue
        inspected.append(str(location))
        if saved_identity == expected_identity:
            matches.append(candidate)
    if not matches:
        raise RuntimeError(
            'No matching Recovery found. Add the previous results as Input; '
            'both ZIP and extracted results/ are supported. '
            f'Found result locations: {inspected}. Refusing to start Stage 1 without KGAT Recovery.'
        )
    if len(matches) != 1:
        raise RuntimeError(f'Multiple recoveries match: {matches}. Set RESTORE_SOURCE explicitly.')
    kind, source = matches[0]
    print('Recovery selected:', source, flush=True)
    if kind == 'zip':
        return source
    target = working / 'kgat-recovery-normalized.zip'
    with ZipFile(target, 'w', compression=ZIP_DEFLATED) as archive:
        for folder in (source, source.parent / 'reports'):
            if folder.is_dir():
                for path in sorted(folder.rglob('*')):
                    if path.is_file():
                        archive.write(path, path.relative_to(source.parent).as_posix())
    return target


def require_baselines(suite):
    """Require complete summaries, unique test seeds, and all per-user outputs."""
    for model in ('mostpop', 'bprmf'):
        if not suite['complete'](model):
            raise RuntimeError(
                f'{model}: incomplete in Recovery. Add the complete previous results; '
                'this notebook only runs KGAT and will not retrain baselines.'
            )
        print(f'{model}: complete (verified)', flush=True)



def require_stage1_inputs(suite):
    """Validate the completed KGAT dependency and the full Stage 1 gamma/seed grid."""
    require_baselines(suite)
    if not suite['complete']('kgat'):
        raise RuntimeError('KGAT is incomplete in Recovery. Use the completed KGAT ZIP before Stage 1.')
    print('kgat: complete (verified)', flush=True)
    import yaml

    model_path = suite['ROOT'] / 'configs/model/tkgat.yaml'
    config = yaml.safe_load(model_path.read_text(encoding='utf-8'))
    if (config.get('name') != 'tkgat' or config.get('time_on', 'interact') != 'interact'
            or config.get('control') or config.get('from_kgat') != 'kgat'):
        raise RuntimeError('Expected the unpermuted Stage 1 configuration from KGAT.')
    if config.get('seeds') != list(range(2019, 2024)):
        raise RuntimeError('Stage 1 requires all five seeds 2019-2023.')
    if config.get('grid') != [{'gamma': 1}, {'gamma': 3}, {'gamma': 10}]:
        raise RuntimeError('Stage 1 requires the complete gamma grid [1, 3, 10].')
    runner = runpy.run_path(str(suite['ROOT'] / 'scripts/run.py'))
    folder = suite['ROOT'] / 'results/runs' / suite['RUN_NAME']
    points, base = runner['tkgat_points'](config, model_path, folder)
    print('Inherited KGAT configuration:', json.dumps(base, sort_keys=True), flush=True)
    print('Stage 1 grid: gamma=[1, 3, 10], seeds=[2019, 2020, 2021, 2022, 2023]; 15 fits.', flush=True)
    return points


input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
if PAYLOAD_DIR is None:
    manifests = sorted(input_root.rglob('payload-manifest.json'))
    if not manifests:
        packages = sorted(input_root.rglob('tkgat-lastfm-latest-kaggle.zip'))
        if len(packages) != 1:
            raise RuntimeError('Add the original code/data Dataset as Input, or set PAYLOAD_DIR.')
        unpacked = working / 'tkgat-payload'
        with ZipFile(packages[0]) as archive:
            for name in archive.namelist():
                path = PurePosixPath(name)
                if path.is_absolute() or '..' in path.parts or '\\' in name:
                    raise ValueError(f'Invalid payload member: {name}')
            archive.extractall(unpacked)
        manifests = [unpacked / 'payload-manifest.json']
    if len({path.read_text(encoding='utf-8') for path in manifests}) != 1:
        raise RuntimeError('Multiple code/data payloads found. Set PAYLOAD_DIR.')
    payload_root = manifests[0].parent
else:
    payload_root = Path(PAYLOAD_DIR)
manifest = json.loads((payload_root / 'payload-manifest.json').read_text(encoding='utf-8'))
bundle = payload_root / 'source.bundle'
if hashlib.sha256(bundle.read_bytes()).hexdigest() != manifest['files_sha256']['source.bundle']:
    raise ValueError('Source bundle hash mismatch')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', str(bundle), str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if commit != manifest['source_commit']:
    raise RuntimeError('Working copy belongs to another payload. Start a fresh session.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
suite = runpy.run_path(str(PROJECT_ROOT / 'scripts/kaggle_lastfm.py'))
suite['prepare'](payload_root)
has_progress = (PROJECT_ROOT / 'results/registry.csv').exists() or any(
    (PROJECT_ROOT / 'results/runs').rglob('*.pt')
)
if not has_progress:
    restore = recovery_archive(input_root, working, suite['identity'](), RESTORE_SOURCE)
    suite['extract_results'](restore)
else:
    print('Keeping existing working progress; Recovery will not overwrite it.', flush=True)
suite['record_identity']()
require_stage1_inputs(suite)
suite['backup'](CHECKPOINT_ZIP)
print('Source snapshot:', commit, flush=True)
print('Recovery verified. T-KGAT Stage 1 is the only model enabled.', flush=True)

## 2. Tiếp tục lưới T-KGAT Stage 1

Log phải xác nhận `kgat: complete (verified)` và hiển thị lưới đầy đủ 15 lượt.
Thứ tự mặc định: γ=1 đủ 5 seed, rồi γ=3, rồi γ=10.
Lượt đã hoàn thành được đọc từ checkpoint; lượt bị ngắt giữa epoch phải train lại.
Mỗi lần gọi hoàn thành tối đa 1 lượt mới rồi lưu ZIP.
Sau đủ 15 lượt, script chọn γ trên validation và chấm test 5 seed của γ đã chọn.
Không tự chạy đối chứng hoán vị hay Stage 2 trong notebook này.

In [ ]:
import torch

require_stage1_inputs(suite)
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU T4 before Save & Run All.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
if NEW_TKGAT_FITS_THIS_SESSION < 1:
    raise ValueError('NEW_TKGAT_FITS_THIS_SESSION must be positive.')

if RUN_TRAINING:
    for step in range(NEW_TKGAT_FITS_THIS_SESSION):
        if suite['complete']('tkgat'):
            print('tkgat: complete. No other model will be started.', flush=True)
            break
        print(f'Running tkgat Stage 1 only; fit budget {step + 1}/{NEW_TKGAT_FITS_THIS_SESSION}', flush=True)
        try:
            suite['execute']([
                'scripts/run.py', '--data', suite['DATA_CONFIG'],
                '--model', 'configs/model/tkgat.yaml', '--device', 'cuda',
                '--resume', '--max-new-runs', '1',
            ])
        finally:
            suite['backup'](CHECKPOINT_ZIP)
else:
    print('Training disabled. Recovery verified only.', flush=True)

suite['backup'](CHECKPOINT_ZIP)
print('tkgat:', 'complete' if suite['complete']('tkgat') else 'pending (more fits needed)', flush=True)
print('Save this output:', CHECKPOINT_ZIP, flush=True)

## 3. Lưu kết quả và chạy phiên sau

Sau khi Version thành công, tải **`tkgat-lastfm-stage1-results.zip`** trong Output.
Thay Recovery cũ bằng ZIP Stage 1 mới nhất trước khi chạy Version kế tiếp.
Giữ Dataset code/data ban đầu. Nếu chỉ giữ 2 lượt/phiên, cần tối đa 8 phiên
(7 phiên đầu mỗi phiên 2 lượt; phiên cuối hoàn tất lượt 15 rồi chọn γ và chấm test).

Khi hoàn tất, `tkgat_summary.json` ghi γ được chọn, Recall/NDCG test trung bình ±
độ lệch chuẩn mẫu trên 5 seed; ZIP giữ kết quả KGAT/baseline và toàn bộ tiến độ Stage 1.
Bước sau là đối chứng hoán vị Stage 1 và/hoặc Stage 2, dùng notebook tương ứng.